# Detailed Checklist — Aircraft Valuation & Similar Asset-Pricing Projects

Markdown checklists — click into a cell and toggle `[ ]` to `[x]`, or just track progress visually. Organized in four parts:

1. **This project's step-by-step checklist**
2. **Transferable checklist** for any similar asset-valuation regression project
3. **Common pitfalls checklist** (including the pandas parsing gotcha and multiplicative-process traps)
4. **Definition-of-done checklist** for the whole project


## Part 1 — This Project's Step-by-Step Checklist

### Step 1 — Load & Inspect
- [ ] Import pandas, numpy, matplotlib, seaborn
- [ ] Load `aircraft_valuation.csv` into `df`
- [ ] Print `df.shape`, `df.head()`, `df.info()`
- [ ] Run `df.describe(include='all').T`
- [ ] Identify every numeric-looking text column (units: hrs, cycles, kg/hr, seats, ADs)


### Step 2 — Data Quality Audit
- [ ] Build a dtype/nunique/missing audit table
- [ ] Check `df.duplicated().sum()`; drop confirmed duplicates
- [ ] Investigate `damage_history`'s null count — confirm it's the `"None"`-as-NaN pandas parsing trap, not real missing data
- [ ] Fix it with `.fillna('None')`, documented with a reason, not a silent patch
- [ ] Normalize `manufacturer` casing


### Step 3 — Unit-Bearing Measurements
- [ ] Strip comma + `" hrs"`/`" cycles"` from all four hour/cycle columns; cast to float
- [ ] Extract numeric seat count from `seat_count`; create `is_freighter` flag
- [ ] Strip comma + `" kg/hr"` from `fuel_burn_kg_per_hr`


### Step 4 — Disguised Missing Values & Flags
- [ ] Clean `airworthiness_directives_open`: strip `" ADs"`, handle `"Not Reported"` sentinel, **add a missing flag**, impute
- [ ] Convert `winglets_installed`/`avionics_upgrade` from Yes/No to 0/1


### Step 5 — Ordinal Encoding & Domain Features
- [ ] Map `airframe_check_status` to an ordinal score
- [ ] Map `damage_history` to an ordinal score
- [ ] Map `paint_condition` to an ordinal score
- [ ] Justify why ordinal (not one-hot) is correct for these three columns
- [ ] Create `age_years`, `engine_overhaul_pct_remaining`, `cycles_per_flight_hour`


### Step 6 — Advanced EDA & the Leakage Audit
- [ ] Target distribution plotted; raw vs. log skewness compared
- [ ] Correlation heatmap built **including** `broker_asking_price_usd` and `recent_comparable_sale_price_usd`
- [ ] Explicit correlation values computed for both market-quote columns
- [ ] Written reasoning for excluding each — noting that a settled comparable sale and a broker's ask carry different degrees of leakage risk
- [ ] VIF computed on the retained feature set
- [ ] Boxplots of target by `lease_status` and `damage_history`
- [ ] Outlier check considered on both raw and log scale, with a note on whether IQR is even meaningful given the wide dynamic range


### Step 7 — Leakage-Safe Encoding
- [ ] Ordinal columns' original text versions dropped after scoring (avoid encoding them twice)
- [ ] Remaining nominal categoricals bucketed by cardinality
- [ ] Train/test split performed **before** any target-encoding statistic
- [ ] Target-encoding maps fit on train only; unseen-category fallback in place
- [ ] Zero `NaN`s confirmed in `X_train`/`X_test`


### Step 8 — Baseline, Linear & Log-Linear Models
- [ ] Mean-predictor baseline scored
- [ ] `LinearRegression` on the raw target scored
- [ ] `LinearRegression` on `log(target)`, predictions exponentiated back to dollars, scored
- [ ] Comparison written: does the log transform help, and does that match the skewness finding from Step 6?


### Step 9 — Tree-Ensemble Models
- [ ] `RandomForestRegressor` (default) scored
- [ ] `GradientBoostingRegressor` (default) scored
- [ ] All models collected into one comparison table, sorted by MAE
- [ ] Interpretability trade-off noted: is the log-linear model "good enough" to prefer over a marginally-better black-box model?


### Step 10 — Cross-Validation & Tuning
- [ ] 5-fold CV MAE computed for the leading candidate
- [ ] Hyperparameter distribution defined
- [ ] `RandomizedSearchCV` run with `cv=5`
- [ ] Tuned model refit and evaluated on the held-out test set


### Step 11 — Evaluation & Diagnostics
- [ ] MAE, RMSE, R², **and MAPE** reported (MAPE is a fair headline metric here — the target never nears zero)
- [ ] Predicted-vs-actual scatterplot
- [ ] Residuals-vs-predicted checked for a funnel pattern; percentage error checked too before concluding it's a problem


### Step 12 — Feature Importance
- [ ] Impurity-based or coefficient-based top-10 plotted
- [ ] Permutation importance top-10 plotted
- [ ] Agreement checked against domain intuition (type & age should dominate)


### Step 13 — Deal-Finding Backtest
- [ ] `edge = model_prediction - broker_asking_price_usd` computed on the test set
- [ ] Flagging threshold defined **before** looking at confirmation results
- [ ] Confirmation check run against `recent_comparable_sale_price_usd` (never a training feature)
- [ ] Confirmed rate compared to the all-test-aircraft baseline rate
- [ ] Caveats written: small/synthetic sample, proxy (not certain) confirmation signal, missing real-world negotiation/counterparty factors


### Step 14 — Persistence & Inference
- [ ] Final model saved with `joblib.dump`
- [ ] Encoding maps, ordinal maps, and column order saved alongside the model
- [ ] `predict_fair_value()` function written, reproducing every training-time transform
- [ ] Function tested on 2–3 made-up aircraft with plausible outputs


### Step 15 — Conclusions
- [ ] Final model choice stated with a one-line justification, including the log-transform decision
- [ ] Expected error stated in both dollars and percentage terms
- [ ] Top 3–5 value drivers named
- [ ] Backtest result stated with at least one reason for caution
- [ ] At least one limitation and one next step named


## Part 2 — Transferable Checklist for Asset-Valuation Regression Projects

### Phase 1 — Understand the Problem
- [ ] Target and any market-quote-style comparison column clearly identified
- [ ] Project goal stated precisely (predict the market's number vs. build an independent estimate)
- [ ] Whether the value-generating process is likely additive or multiplicative considered up front (does value change by a fixed amount or a percentage over the relevant driver?)


### Phase 2 — Data Acquisition & Audit
- [ ] Shape, dtypes, nulls, duplicates all checked
- [ ] **Explicitly checked for a pandas default-NA-string collision** on any column that might have a "None"/"NA"/"Null"-named category
- [ ] Every combined-format or unit-bearing text column identified with a parsing plan
- [ ] Category-casing inconsistencies checked


### Phase 3 — Feature Engineering
- [ ] Unit-bearing strings converted to numerics, idempotently
- [ ] Naturally-ordered categories identified and ordinal-encoded (not one-hot)
- [ ] Ratio/derived features constructed where domain logic supports it (age, utilization rate, remaining-life fraction)
- [ ] Missing-value strategy chosen per column, with a flag column preserved where relevant


### Phase 4 — Target Transform Check
- [ ] Raw target skewness computed
- [ ] Log-target skewness computed and compared
- [ ] If the target is strictly positive with a wide dynamic range, a log-linear model included in the model comparison


### Phase 5 — The Market-Quote Leakage Audit
- [ ] Every market-quote-style column identified and listed
- [ ] Correlation of each with the target checked
- [ ] For each: reasoned about whether it represents a settled fact, an opinion/quote, or a genuinely independent signal — and made an inclusion/exclusion decision accordingly
- [ ] VIF checked on the retained feature set


### Phase 6 — Encoding & Splitting
- [ ] Ordinal columns scored numerically; original text versions excluded from further encoding
- [ ] Remaining nominal categoricals partitioned by cardinality
- [ ] Split performed before any target-dependent computation
- [ ] Target encoding fit on train only, with an unseen-category fallback


### Phase 7 — Modeling
- [ ] Baseline, linear, log-linear, AND tree-ensemble models all compared
- [ ] No model family assumed to win in advance
- [ ] Interpretability considered as a factor, not just raw accuracy, when picking a "final" model for a report-facing use case


### Phase 8 — Evaluation, Interpretation & Backtesting
- [ ] Metrics matched to the target's scale (MAPE favored for strictly-positive, wide-range targets)
- [ ] Feature importance computed via two independent methods
- [ ] If backtesting against a market/proxy quote: confirmation signal's reliability itself assessed (is it a certain outcome or noisy proxy evidence?)
- [ ] Results reported with explicit statistical caution


### Phase 9 — Communicate & Persist
- [ ] Findings summarized in plain language with a stated confidence level
- [ ] Model + preprocessing artifacts (including ordinal maps) persisted together
- [ ] Limitations and next steps explicitly named


## Part 3 — Common Pitfalls Checklist

- [ ] **Pandas default-NA-string collision** — a genuine category (`"None"`, `"NA"`, etc.) silently converted to `NaN` on load, inflating an apparent missing-data problem
- [ ] **Ordinal categories treated as one-hot** — throwing away a genuine, useful ranking (e.g. maintenance-check freshness) by encoding it as unordered dummies
- [ ] **Ignoring a multiplicative process** — fitting a plain linear model to a strictly-positive, wide-range, percentage-driven target without ever testing a log transform
- [ ] **Market-quote leakage treated as one-size-fits-all** — excluding every high-correlation "market" column without reasoning about whether it's a settled fact, an opinion, or a genuinely independent signal
- [ ] **MAE reported alone on a wide-dynamic-range target** — hiding whether errors are proportionally consistent or concentrated in one segment
- [ ] **Backtest confirmation signal treated as ground truth** — a proxy outcome (a comparable sale, a later appraisal) isn't the same as a certain, resolved result
- [ ] **Small-sample overconfidence** — treating a good-looking backtest confirmation rate as proof of a real pricing edge
- [ ] **Assuming tree ensembles always win** — skipping the (log-)linear comparison because "ensembles are usually better"
- [ ] **Discarding missingness signal** — imputing a disguised-missing sentinel without keeping a flag column
- [ ] **Model without its preprocessing** — persisting a model without the encoding maps (including ordinal maps) needed to use it on new data


## Part 4 — Definition-of-Done Checklist for the Whole Project

- [ ] Every raw column is either numeric, properly encoded (one-hot, target, or ordinal as appropriate), or intentionally dropped with a stated reason
- [ ] The pandas default-NA-string check has been explicitly performed and documented
- [ ] No `NaN` values remain anywhere in the final training/test feature matrices
- [ ] Every market-quote-style column has an explicit, written inclusion/exclusion decision with reasoning
- [ ] Raw-target and log-target models were both tried
- [ ] At least one linear-family and one tree-ensemble model trained and fairly compared
- [ ] Final model selected with a written justification that considers both accuracy and interpretability
- [ ] Cross-validated performance estimate reported alongside single-split test metrics
- [ ] Two independent feature-importance methods agree on (most of) the top drivers
- [ ] If a decision rule was backtested: the confirmation signal's reliability was assessed, and results were compared to a meaningful baseline
- [ ] Model and preprocessing artifacts (including ordinal/target-encoding maps) are persisted and reloadable
- [ ] A plain-language summary exists, including explicit caveats about synthetic/limited data
